# Analytics Recipes

## Quick notes
### What this covers
- Customer analytics: RFM segmentation (`NTILE`), customer lifetime value, basket size and AOV.
- Product analytics: Pareto / ABC analysis, market basket pairs (self-join), moving top-sellers.
- Distribution: median and percentiles (window and `OFFSET` methods), cumulative distribution, bucketing with `CASE` and with a ranges table.
- Business reporting: ratio-to-total, growth rate, CAGR, contribution to change, price/volume/mix bridge, weighted averages, budget vs actual variance, and an executive-summary query built from CTEs.
### Key points
- Almost every recipe is: build a clean base view once, aggregate to the right grain, then add one window function.
- Revenue = `SUM(qty * unit_price)`, never `SUM(unit_price) * SUM(qty)`. Compute at line level first.
- Window `SUM(...) OVER ()` gives the grand total on every row, so share-of-total needs no self-join.
- Ratios of sums are not averages of ratios. Weight by the denominator (weighted average).
- Fixed "as of" dates make reports repeatable. Use a literal date or a parameter row, not "today".
### Gotchas (the part worth revising twice)
- Integer division: `100 * a / b` with integers truncates. Multiply by `1.0` or `100.0`.
- `NTILE(n)` splits by row position, so tied values can land in different buckets. Add a unique tiebreaker and know that bucket sizes differ by at most one.
- For "recent is good" scores, sort recency descending so the most recent customers get the top bucket.
- Cumulative share needs a deterministic order (`ORDER BY revenue DESC, id`), otherwise ties make the running total unstable. The default frame with `RANGE` also lumps tied rows together.
- A median of an even count is the average of the two middle rows. `OFFSET` method: `LIMIT 2 - n % 2 OFFSET (n - 1) / 2`.
- Range joins need half-open intervals (`lo <= x AND x < hi`), or a value on the boundary lands in two buckets.
- A growth rate from zero is undefined. Guard with `NULLIF(prev, 0)`.
- A price/volume bridge only adds up if you define the effects consistently (here: price effect uses new volume, volume effect uses old price).
- `power()`, `ln()`, `exp()` need SQLite built with math functions (available from 3.35 when compiled in; most builds include them, but check yours).
### When it matters
- Marketing segments, sales reviews, finance variance packs, inventory focus (A items), "what drove the change" questions, board summaries.
### One-line recap example
```sql
SELECT category, SUM(revenue) AS rev, ROUND(100.0 * SUM(revenue) / SUM(SUM(revenue)) OVER (), 1) AS pct FROM v_lines GROUP BY category;
```

## 01. Seed data: a small shop
- Six products in three categories, eight customers, 15 orders across 2023 and 2024, a yearly budget and a short history of annual revenue.
- Prices rose a little between years (Headphones 55 to 60, Lamp 28 to 30), which the price bridge uses later. The Cable is new in 2024.
- For joins used here see `02-Querying/01-Joins`; for window functions see `03-Window-Functions/01-Window-Functions`.

In [1]:
# setup: an in-memory SQLite database and a %%sql cell magic that prints result tables
import sqlite3
from IPython.core.magic import register_cell_magic

con = sqlite3.connect(":memory:", isolation_level=None)   # autocommit, so BEGIN/COMMIT work as written
con.execute("PRAGMA foreign_keys = ON")

def _table(cols, rows):
    cells = [[("NULL" if v is None else str(v)) for v in r] for r in rows]
    widths = [max([len(c)] + [len(r[i]) for r in cells]) for i, c in enumerate(cols)]
    line = lambda vals: " | ".join(v.ljust(w) for v, w in zip(vals, widths))
    print(line(cols)); print("-+-".join("-" * w for w in widths))
    for r in cells: print(line(r))
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''})\n")

def _statements(text):
    """Split a cell into statements (handles several per line, triggers, quoted semicolons)."""
    out, start = [], 0
    for i, ch in enumerate(text):
        if ch == ";" and sqlite3.complete_statement(text[start:i + 1]):
            out.append(text[start:i + 1].strip()); start = i + 1
    if text[start:].strip():
        out.append(text[start:].strip())
    return out

@register_cell_magic
def sql(line, cell):
    """Run every statement in the cell; print rows for statements that return them."""
    expect_error = "expect_error" in line
    for stmt in _statements(cell):
        try:
            cur = con.execute(stmt)
            if cur.description:
                _table([d[0] for d in cur.description], cur.fetchall())
        except sqlite3.Error as exc:
            if not expect_error:
                raise
            print(f"ERROR: {exc}\n")

In [2]:
%%sql
CREATE TABLE products (id INTEGER PRIMARY KEY, name TEXT, category TEXT, cost REAL);
INSERT INTO products VALUES
 (1,'Pen','Stationery',1),(2,'Notebook','Stationery',2),(3,'Mug','Home',4),
 (4,'Lamp','Home',18),(5,'Headphones','Tech',35),(6,'Cable','Tech',3);
CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT);
INSERT INTO customers VALUES (1,'Asha'),(2,'Ben'),(3,'Cara'),(4,'Dev'),(5,'Eli'),(6,'Fay'),(7,'Gus'),(8,'Hana');
CREATE TABLE orders (id INTEGER PRIMARY KEY, customer_id INTEGER REFERENCES customers(id), order_date TEXT);
INSERT INTO orders VALUES
 (1,1,'2023-02-10'),(2,1,'2023-11-05'),(3,1,'2024-06-20'),(4,1,'2024-12-01'),
 (5,2,'2023-03-15'),(6,2,'2024-09-10'),(7,3,'2023-07-01'),
 (8,4,'2024-01-12'),(9,4,'2024-05-30'),(10,4,'2024-11-15'),
 (11,5,'2023-05-05'),(12,6,'2024-03-03'),(13,6,'2024-10-10'),
 (14,7,'2024-12-20'),(15,8,'2023-12-25');
CREATE TABLE order_items (order_id INTEGER REFERENCES orders(id), product_id INTEGER REFERENCES products(id), qty INTEGER, unit_price REAL, PRIMARY KEY (order_id, product_id));
INSERT INTO order_items VALUES
 (1,5,1,55),(1,1,5,2),
 (2,2,2,5),(2,3,1,8),
 (3,4,1,30),(3,6,2,10),
 (4,5,1,60),
 (5,3,2,8),
 (6,2,3,5),(6,1,10,2),
 (7,4,2,28),
 (8,6,1,10),(8,1,2,2),
 (9,5,1,60),(9,6,1,10),
 (10,3,1,8),(10,2,1,5),
 (11,1,3,2),
 (12,5,2,60),
 (13,4,1,30),(13,3,1,8),(13,6,1,10),
 (14,2,2,5),(14,1,4,2),
 (15,3,3,8),(15,4,1,28);
CREATE TABLE budget_2024 (category TEXT PRIMARY KEY, budget REAL);
INSERT INTO budget_2024 VALUES ('Stationery',60),('Home',150),('Tech',400);
CREATE TABLE annual_revenue (year INTEGER PRIMARY KEY, revenue REAL);
INSERT INTO annual_revenue VALUES (2020,1000),(2021,1180),(2022,1250),(2023,1500),(2024,1760);
SELECT COUNT(*) AS orders, (SELECT COUNT(*) FROM order_items) AS lines FROM orders;

orders | lines
-------+------
15     | 26   
(1 row)



## 02. The base views: one clean grain
- `v_lines`: one row per order line with revenue and cost already computed. Everything else builds on it.
- `v_orders`: one row per order with total, item count and units. Aggregating at the right grain first avoids the classic double-counting join mistake.

In [3]:
%%sql
CREATE VIEW v_lines AS
SELECT o.id AS order_id, o.customer_id, o.order_date,
       CAST(strftime('%Y', o.order_date) AS INTEGER) AS year,
       CAST(strftime('%m', o.order_date) AS INTEGER) AS month,
       i.product_id, p.name AS product, p.category, i.qty, i.unit_price,
       i.qty * i.unit_price AS revenue, i.qty * p.cost AS cost_total
FROM orders o JOIN order_items i ON i.order_id = o.id JOIN products p ON p.id = i.product_id;
CREATE VIEW v_orders AS
SELECT order_id, customer_id, order_date, year,
       COUNT(*) AS lines, SUM(qty) AS units, SUM(revenue) AS total
FROM v_lines GROUP BY order_id;
SELECT * FROM v_orders ORDER BY order_id;

order_id | customer_id | order_date | year | lines | units | total
---------+-------------+------------+------+-------+-------+------
1        | 1           | 2023-02-10 | 2023 | 2     | 6     | 65.0 
2        | 1           | 2023-11-05 | 2023 | 2     | 3     | 18.0 
3        | 1           | 2024-06-20 | 2024 | 2     | 3     | 50.0 
4        | 1           | 2024-12-01 | 2024 | 1     | 1     | 60.0 
5        | 2           | 2023-03-15 | 2023 | 1     | 2     | 16.0 
6        | 2           | 2024-09-10 | 2024 | 2     | 13    | 35.0 
7        | 3           | 2023-07-01 | 2023 | 1     | 2     | 56.0 
8        | 4           | 2024-01-12 | 2024 | 2     | 3     | 14.0 
9        | 4           | 2024-05-30 | 2024 | 2     | 2     | 70.0 
10       | 4           | 2024-11-15 | 2024 | 2     | 2     | 13.0 
11       | 5           | 2023-05-05 | 2023 | 1     | 3     | 6.0  
12       | 6           | 2024-03-03 | 2024 | 1     | 2     | 120.0
13       | 6           | 2024-10-10 | 2024 | 3     | 3     | 4

## 03. Basket size and average order value (AOV)
- Basket size = units (or lines) per order. AOV = revenue / orders.
- Compute AOV as `SUM(total) / COUNT(*)` over the order-level view. Dividing line-level revenue by line count gives the wrong number (average line value).

In [4]:
%%sql
SELECT year, COUNT(*) AS orders, SUM(total) AS revenue,
       ROUND(1.0 * SUM(total) / COUNT(*), 2) AS aov,
       ROUND(1.0 * SUM(units) / COUNT(*), 2) AS avg_units_per_order,
       ROUND(1.0 * SUM(lines) / COUNT(*), 2) AS avg_lines_per_order
FROM v_orders GROUP BY year ORDER BY year;
-- Wrong AOV: average of lines, not orders
SELECT year, ROUND(AVG(revenue), 2) AS avg_line_value_not_aov FROM v_lines GROUP BY year ORDER BY year;

year | orders | revenue | aov   | avg_units_per_order | avg_lines_per_order
-----+--------+---------+-------+---------------------+--------------------
2023 | 6      | 213.0   | 35.5  | 3.33                | 1.5                
2024 | 9      | 428.0   | 47.56 | 3.89                | 1.89               
(2 rows)

year | avg_line_value_not_aov
-----+-----------------------
2023 | 23.67                 
2024 | 25.18                 
(2 rows)



## 04. Ratio-to-total with a window SUM
- `SUM(x) OVER ()` = grand total on every row. `SUM(x) OVER (PARTITION BY y)` = total within a group.
- You can even wrap an aggregate: `SUM(SUM(revenue)) OVER ()` sums the already grouped values.
- Use `100.0 *` to avoid integer division.

In [5]:
%%sql
SELECT category, SUM(revenue) AS revenue,
       ROUND(100.0 * SUM(revenue) / SUM(SUM(revenue)) OVER (), 1) AS pct_of_all
FROM v_lines GROUP BY category ORDER BY revenue DESC;
-- share within each year
SELECT year, category, SUM(revenue) AS revenue,
       ROUND(100.0 * SUM(revenue) / SUM(SUM(revenue)) OVER (PARTITION BY year), 1) AS pct_of_year
FROM v_lines GROUP BY year, category ORDER BY year, category;

category   | revenue | pct_of_all
-----------+---------+-----------
Tech       | 345.0   | 53.8      
Home       | 208.0   | 32.4      
Stationery | 88.0    | 13.7      
(3 rows)

year | category   | revenue | pct_of_year
-----+------------+---------+------------
2023 | Home       | 132.0   | 62.0       
2023 | Stationery | 26.0    | 12.2       
2023 | Tech       | 55.0    | 25.8       
2024 | Home       | 76.0    | 17.8       
2024 | Stationery | 62.0    | 14.5       
2024 | Tech       | 290.0   | 67.8       
(6 rows)



## 05. Growth rate: year over year and month over month
- Growth % = (this - previous) / previous. Get "previous" with `LAG` (see `03-Window-Functions/01-Window-Functions`).
- `NULLIF(prev, 0)` stops divide-by-zero. The first period has no previous value, so growth is NULL.
- Months with no sales do not appear in a `GROUP BY`. `LAG` then compares to the previous month that has data, not the previous calendar month. Use a calendar table when you need exact months (section 18 builds one).

In [6]:
%%sql
SELECT year, ROUND(SUM(revenue), 2) AS revenue,
       ROUND(100.0 * (SUM(revenue) - LAG(SUM(revenue)) OVER (ORDER BY year)) / NULLIF(LAG(SUM(revenue)) OVER (ORDER BY year), 0), 1) AS yoy_pct
FROM v_lines GROUP BY year ORDER BY year;
SELECT strftime('%Y-%m', order_date) AS ym, SUM(revenue) AS revenue,
       SUM(revenue) - LAG(SUM(revenue)) OVER (ORDER BY strftime('%Y-%m', order_date)) AS change_vs_prev_row
FROM v_lines WHERE year = 2024 GROUP BY ym ORDER BY ym;

year | revenue | yoy_pct
-----+---------+--------
2023 | 213.0   | NULL   
2024 | 428.0   | 100.9  
(2 rows)

ym      | revenue | change_vs_prev_row
--------+---------+-------------------
2024-01 | 14.0    | NULL              
2024-03 | 120.0   | 106.0             
2024-05 | 70.0    | -50.0             
2024-06 | 50.0    | -20.0             
2024-09 | 35.0    | -15.0             
2024-10 | 48.0    | 13.0              
2024-11 | 13.0    | -35.0             
2024-12 | 78.0    | 65.0              
(8 rows)



## 06. CAGR (compound annual growth rate)
- CAGR = (end / start) ^ (1 / years) - 1. It is the steady yearly rate that would take you from start to end.
- With `power()` (math functions). Years = last year - first year, not the number of rows.
- Compare with the plain average of yearly growth rates: it is higher, because compounding is not linear.

In [7]:
%%sql
WITH ends AS (
    SELECT (SELECT revenue FROM annual_revenue ORDER BY year LIMIT 1)      AS start_rev,
           (SELECT revenue FROM annual_revenue ORDER BY year DESC LIMIT 1) AS end_rev,
           (SELECT MAX(year) - MIN(year) FROM annual_revenue)              AS years
)
SELECT start_rev, end_rev, years,
       ROUND(100.0 * (power(1.0 * end_rev / start_rev, 1.0 / years) - 1), 2) AS cagr_pct
FROM ends;
WITH g AS (
    SELECT year, 100.0 * (revenue - LAG(revenue) OVER (ORDER BY year)) / LAG(revenue) OVER (ORDER BY year) AS growth
    FROM annual_revenue
)
SELECT ROUND(AVG(growth), 2) AS simple_avg_of_growth_pct FROM g;

start_rev | end_rev | years | cagr_pct
----------+---------+-------+---------
1000.0    | 1760.0  | 4     | 15.18   
(1 row)

simple_avg_of_growth_pct
------------------------
15.32                   
(1 row)



## 07. Contribution to change
- "Revenue grew by 215. Where did it come from?" Contribution = category change / total change.
- Pivot-style with `SUM(CASE ...)` (or `FILTER`) because SQLite has no PIVOT.
- Contributions add up to 100%. A category that fell (Home) shows a negative contribution, so another one (Tech) can contribute more than 100%.

In [8]:
%%sql
WITH by_cat AS (
    SELECT category,
           SUM(revenue) FILTER (WHERE year = 2023) AS y2023,
           SUM(revenue) FILTER (WHERE year = 2024) AS y2024
    FROM v_lines GROUP BY category
)
SELECT category, COALESCE(y2023, 0) AS y2023, COALESCE(y2024, 0) AS y2024,
       COALESCE(y2024, 0) - COALESCE(y2023, 0) AS change,
       ROUND(100.0 * (COALESCE(y2024, 0) - COALESCE(y2023, 0)) / SUM(COALESCE(y2024, 0) - COALESCE(y2023, 0)) OVER (), 1) AS pct_of_total_change
FROM by_cat ORDER BY change DESC, category;

category   | y2023 | y2024 | change | pct_of_total_change
-----------+-------+-------+--------+--------------------
Tech       | 55.0  | 290.0 | 235.0  | 109.3              
Stationery | 26.0  | 62.0  | 36.0   | 16.7               
Home       | 132.0 | 76.0  | -56.0  | -26.0              
(3 rows)



## 08. Price / volume / new-product bridge
- A bridge explains the change in revenue in pieces that add up exactly.
- Price effect = (new price - old price) x new quantity. Volume effect = (new qty - old qty) x old price. For a product that did not exist last year, the whole revenue is a "new product" effect.
- Average price per product per year is `revenue / qty` (a weighted average price).
- The last row proves the pieces sum to the total change.

In [9]:
%%sql
WITH p AS (
    SELECT product,
           COALESCE(SUM(qty) FILTER (WHERE year = 2023), 0)     AS q23,
           COALESCE(SUM(qty) FILTER (WHERE year = 2024), 0)     AS q24,
           COALESCE(SUM(revenue) FILTER (WHERE year = 2023), 0) AS r23,
           COALESCE(SUM(revenue) FILTER (WHERE year = 2024), 0) AS r24
    FROM v_lines GROUP BY product
),
b AS (
    SELECT product, r23, r24,
           CASE WHEN q23 = 0 THEN 0 ELSE ROUND((1.0 * r24 / q24 - 1.0 * r23 / q23) * q24, 2) END AS price_effect,
           CASE WHEN q23 = 0 THEN 0 ELSE ROUND((q24 - q23) * 1.0 * r23 / q23, 2) END           AS volume_effect,
           CASE WHEN q23 = 0 THEN r24 ELSE 0 END                                                AS new_product_effect
    FROM p
)
SELECT * FROM (
SELECT 0 AS o, product, r23, r24, r24 - r23 AS change, price_effect, volume_effect, new_product_effect,
       ROUND(price_effect + volume_effect + new_product_effect, 2) AS explained
FROM b
UNION ALL
SELECT 1, 'TOTAL', SUM(r23), SUM(r24), SUM(r24 - r23), SUM(price_effect), SUM(volume_effect), SUM(new_product_effect),
       ROUND(SUM(price_effect + volume_effect + new_product_effect), 2)
FROM b
) ORDER BY o, product;

o | product    | r23   | r24   | change | price_effect | volume_effect | new_product_effect | explained
--+------------+-------+-------+--------+--------------+---------------+--------------------+----------
0 | Cable      | 0     | 50.0  | 50.0   | 0            | 0             | 50.0               | 50.0     
0 | Headphones | 55.0  | 240.0 | 185.0  | 20.0         | 165.0         | 0                  | 185.0    
0 | Lamp       | 84.0  | 60.0  | -24.0  | 4.0          | -28.0         | 0                  | -24.0    
0 | Mug        | 48.0  | 16.0  | -32.0  | 0.0          | -32.0         | 0                  | -32.0    
0 | Notebook   | 10.0  | 30.0  | 20.0   | 0.0          | 20.0          | 0                  | 20.0     
0 | Pen        | 16.0  | 32.0  | 16.0   | 0.0          | 16.0          | 0                  | 16.0     
1 | TOTAL      | 213.0 | 428.0 | 215.0  | 24.0         | 141.0         | 50.0               | 215.0    
(7 rows)



## 09. Weighted averages
- A simple average of prices treats a pen sold once like a pen sold a thousand times. The weighted average price is `SUM(price * qty) / SUM(qty)`.
- Same idea for margin %: total margin / total revenue, not the average of per-line margin %.

In [10]:
%%sql
SELECT category,
       ROUND(AVG(unit_price), 2)               AS simple_avg_price,
       ROUND(1.0 * SUM(revenue) / SUM(qty), 2) AS weighted_avg_price,
       ROUND(AVG(100.0 * (revenue - cost_total) / revenue), 1) AS avg_of_line_margin_pct,
       ROUND(100.0 * (SUM(revenue) - SUM(cost_total)) / SUM(revenue), 1) AS true_margin_pct
FROM v_lines GROUP BY category ORDER BY category;

category   | simple_avg_price | weighted_avg_price | avg_of_line_margin_pct | true_margin_pct
-----------+------------------+--------------------+------------------------+----------------
Home       | 17.33            | 16.0               | 44.6                   | 41.3           
Stationery | 3.33             | 2.75               | 54.4                   | 54.5           
Tech       | 34.38            | 34.5               | 55.2                   | 44.9           
(3 rows)



## 10. Bucketing with CASE
- Turn a number into a label with `CASE`. Conditions are tested top to bottom, so order from the smallest band up and the first match wins.
- Then group by the label. Remember empty buckets do not show up (next section fixes that).

In [11]:
%%sql
SELECT CASE WHEN total < 20  THEN '1 small  (< 20)'
            WHEN total < 60  THEN '2 medium (20-59)'
            WHEN total < 100 THEN '3 large  (60-99)'
            ELSE                  '4 jumbo  (100+)' END AS band,
       COUNT(*) AS orders, SUM(total) AS revenue
FROM v_orders GROUP BY band ORDER BY band;

band             | orders | revenue
-----------------+--------+--------
1 small  (< 20)  | 6      | 85.0   
2 medium (20-59) | 5      | 241.0  
3 large  (60-99) | 3      | 195.0  
4 jumbo  (100+)  | 1      | 120.0  
(4 rows)



## 11. Bucketing with a ranges table
- Put the bands in a table so a business user can change them without editing SQL. Join on `lo <= x AND x < hi`.
- `LEFT JOIN` from the bands table keeps empty bands at 0.
- Gotcha: overlapping or closed-both-ends ranges (`BETWEEN`) double count a value sitting on a boundary. Use half-open ranges.

In [12]:
%%sql
CREATE TABLE bands (label TEXT PRIMARY KEY, lo REAL NOT NULL, hi REAL NOT NULL);
INSERT INTO bands VALUES ('small',0,20),('medium',20,60),('large',60,100),('jumbo',100,1000000),('mega',1000000,1e18);
SELECT b.label, b.lo, b.hi, COUNT(o.order_id) AS orders, COALESCE(SUM(o.total), 0) AS revenue
FROM bands b LEFT JOIN v_orders o ON o.total >= b.lo AND o.total < b.hi
GROUP BY b.label ORDER BY b.lo;

label  | lo        | hi        | orders | revenue
-------+-----------+-----------+--------+--------
small  | 0.0       | 20.0      | 6      | 85.0   
medium | 20.0      | 60.0      | 5      | 241.0  
large  | 60.0      | 100.0     | 3      | 195.0  
jumbo  | 100.0     | 1000000.0 | 1      | 120.0  
mega   | 1000000.0 | 1e+18     | 0      | 0      
(5 rows)



## 12. Pareto / ABC analysis (cumulative share)
- Rank items by revenue, take the running total, divide by the grand total. Class A = the first items that make up the first 70%, B = next 20%, C = the tail.
- The running total uses `ROWS` (not the default `RANGE`) and a unique tiebreaker.
- The class is decided by the cumulative share *before* the item is added (`cum - revenue`), so the item that crosses the line still belongs to the higher class.

In [13]:
%%sql
WITH rev AS (
    SELECT product_id, product, SUM(revenue) AS revenue FROM v_lines GROUP BY product_id
),
cum AS (
    SELECT product, revenue,
           SUM(revenue) OVER (ORDER BY revenue DESC, product_id ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running,
           SUM(revenue) OVER () AS grand
    FROM rev
)
SELECT product, revenue,
       ROUND(100.0 * revenue / grand, 1) AS pct,
       ROUND(100.0 * running / grand, 1) AS cum_pct,
       CASE WHEN 100.0 * (running - revenue) / grand < 70 THEN 'A'
            WHEN 100.0 * (running - revenue) / grand < 90 THEN 'B' ELSE 'C' END AS abc_class
FROM cum ORDER BY revenue DESC, product;

product    | revenue | pct  | cum_pct | abc_class
-----------+---------+------+---------+----------
Headphones | 295.0   | 46.0 | 46.0    | A        
Lamp       | 144.0   | 22.5 | 68.5    | A        
Mug        | 64.0    | 10.0 | 78.5    | A        
Cable      | 50.0    | 7.8  | 86.3    | B        
Pen        | 48.0    | 7.5  | 93.8    | B        
Notebook   | 40.0    | 6.2  | 100.0   | C        
(6 rows)



## 13. RFM segmentation with NTILE
- RFM scores each customer on Recency (days since last order), Frequency (orders) and Monetary (total spend).
- `NTILE(4)` gives each customer a score 1 to 4 on each. For recency we sort descending so the most recent customers get 4 (higher is always better).
- Ties (many customers have 1 order) are split by row position, so always add a tiebreaker to the window `ORDER BY` to keep results stable.
- Look at the four one-order customers (Cara, Gus, Hana, Eli): their frequency scores differ (1 or 2) only because `NTILE` splits the tie using the tiebreaker (monetary). That is the tie gotcha in action, so keep the segment rules coarse.
- Segment with a `CASE` on the scores. The "as of" date is a literal so the report never changes.

In [14]:
%%sql
CREATE VIEW v_rfm AS
WITH base AS (
    SELECT c.id, c.name,
           CAST(julianday('2024-12-31') - julianday(MAX(o.order_date)) AS INTEGER) AS recency_days,
           COUNT(*) AS frequency, SUM(o.total) AS monetary
    FROM customers c JOIN v_orders o ON o.customer_id = c.id GROUP BY c.id
)
SELECT id, name, recency_days, frequency, monetary,
       NTILE(4) OVER (ORDER BY recency_days DESC, id) AS r,
       NTILE(4) OVER (ORDER BY frequency, monetary, id) AS f,
       NTILE(4) OVER (ORDER BY monetary, id) AS m
FROM base;
SELECT name, recency_days, frequency, monetary, r, f, m,
       CASE WHEN r >= 3 AND f >= 3 THEN 'Champion'
            WHEN r >= 3 AND f <= 2 THEN 'New / promising'
            WHEN r <= 2 AND f >= 3 THEN 'At risk (was loyal)'
            ELSE 'Hibernating' END AS segment
FROM v_rfm ORDER BY r + f + m DESC, id;

name | recency_days | frequency | monetary | r | f | m | segment            
-----+--------------+-----------+----------+---+---+---+--------------------
Asha | 30           | 4         | 193.0    | 4 | 4 | 4 | Champion           
Dev  | 46           | 3         | 97.0     | 3 | 4 | 3 | Champion           
Fay  | 82           | 2         | 168.0    | 3 | 3 | 4 | Champion           
Ben  | 112          | 2         | 51.0     | 2 | 3 | 2 | At risk (was loyal)
Cara | 549          | 1         | 56.0     | 1 | 2 | 3 | Hibernating        
Gus  | 11           | 1         | 18.0     | 4 | 1 | 1 | New / promising    
Hana | 372          | 1         | 52.0     | 2 | 2 | 2 | Hibernating        
Eli  | 606          | 1         | 6.0      | 1 | 1 | 1 | Hibernating        
(8 rows)



## 14. Customer lifetime value (CLV)
- Historic CLV = total margin a customer has produced so far (revenue minus cost). Margin is a better base than revenue.
- A naive projection: yearly margin x an assumed future lifetime (here 3 years). Yearly margin = margin / years active, with years active floored at 1 so one-order customers are not blown up.
- Treat this as a ranking tool, not a forecast. Real CLV models add churn and discounting.

In [15]:
%%sql
SELECT c.name, COUNT(DISTINCT l.order_id) AS orders, ROUND(SUM(l.revenue), 2) AS revenue,
       ROUND(SUM(l.revenue - l.cost_total), 2) AS margin,
       MIN(l.order_date) AS first_order, MAX(l.order_date) AS last_order,
       ROUND(MAX(1.0, (julianday(MAX(l.order_date)) - julianday(MIN(l.order_date))) / 365.0), 2) AS years_active,
       ROUND(SUM(l.revenue - l.cost_total) / MAX(1.0, (julianday(MAX(l.order_date)) - julianday(MIN(l.order_date))) / 365.0) * 3, 1) AS projected_3yr_margin
FROM customers c JOIN v_lines l ON l.customer_id = c.id
GROUP BY c.id ORDER BY margin DESC, c.name;

name | orders | revenue | margin | first_order | last_order | years_active | projected_3yr_margin
-----+--------+---------+--------+-------------+------------+--------------+---------------------
Asha | 4      | 193.0   | 86.0   | 2023-02-10  | 2024-12-01 | 1.81         | 142.7               
Fay  | 2      | 168.0   | 73.0   | 2024-03-03  | 2024-10-10 | 1.0          | 219.0               
Dev  | 3      | 97.0    | 48.0   | 2024-01-12  | 2024-11-15 | 1.0          | 144.0               
Ben  | 2      | 51.0    | 27.0   | 2023-03-15  | 2024-09-10 | 1.49         | 54.2                
Hana | 1      | 52.0    | 22.0   | 2023-12-25  | 2023-12-25 | 1.0          | 66.0                
Cara | 1      | 56.0    | 20.0   | 2023-07-01  | 2023-07-01 | 1.0          | 60.0                
Gus  | 1      | 18.0    | 10.0   | 2024-12-20  | 2024-12-20 | 1.0          | 30.0                
Eli  | 1      | 6.0     | 3.0    | 2023-05-05  | 2023-05-05 | 1.0          | 9.0                 
(8 rows)



## 15. Market basket: which products are bought together?
- Self-join the order lines on the same order, with `a.product_id < b.product_id` so each pair is counted once and never paired with itself.
- Support = share of orders containing the pair. Confidence of A to B = orders with both / orders with A.
- Pair counts explode quickly on big data (items squared per order). Filter early and add an index on `order_id`.

In [16]:
%%sql
WITH pairs AS (
    SELECT a.product_id AS pa, b.product_id AS pb, COUNT(*) AS together
    FROM order_items a JOIN order_items b ON a.order_id = b.order_id AND a.product_id < b.product_id
    GROUP BY a.product_id, b.product_id
),
cnt AS (SELECT product_id, COUNT(*) AS orders_with FROM order_items GROUP BY product_id)
SELECT p1.name AS item_a, p2.name AS item_b, together,
       ROUND(100.0 * together / (SELECT COUNT(*) FROM orders), 1) AS support_pct,
       ROUND(100.0 * together / ca.orders_with, 1) AS confidence_a_to_b_pct
FROM pairs JOIN products p1 ON p1.id = pairs.pa JOIN products p2 ON p2.id = pairs.pb
JOIN cnt ca ON ca.product_id = pairs.pa
ORDER BY together DESC, item_a, item_b;

item_a     | item_b     | together | support_pct | confidence_a_to_b_pct
-----------+------------+----------+-------------+----------------------
Lamp       | Cable      | 2        | 13.3        | 50.0                 
Mug        | Lamp       | 2        | 13.3        | 40.0                 
Notebook   | Mug        | 2        | 13.3        | 50.0                 
Pen        | Notebook   | 2        | 13.3        | 40.0                 
Headphones | Cable      | 1        | 6.7         | 25.0                 
Mug        | Cable      | 1        | 6.7         | 20.0                 
Pen        | Cable      | 1        | 6.7         | 20.0                 
Pen        | Headphones | 1        | 6.7         | 20.0                 
(8 rows)



## 16. Median and percentiles: window method and OFFSET method
- SQLite has no `MEDIAN()` or `PERCENTILE_CONT`. Two ways:
- Window: number the rows, count them, keep the middle one or two, average them.
- `OFFSET`: sort, skip `(n - 1) / 2` rows, take 1 row (odd n) or 2 rows (even n) with `LIMIT 2 - n % 2`, then `AVG`.
- Both give the same answer. The `OFFSET` one is short but cannot do many percentiles in one pass.
- Nearest-rank percentile p: the row at position `ceil(p * n)`.

In [17]:
%%sql
WITH r AS (SELECT total, ROW_NUMBER() OVER (ORDER BY total, order_id) AS rn, COUNT(*) OVER () AS n FROM v_orders)
SELECT AVG(total) AS median_window FROM r WHERE rn IN ((n + 1) / 2, (n + 2) / 2);
SELECT AVG(total) AS median_offset FROM (
    SELECT total FROM v_orders ORDER BY total
    LIMIT 2 - (SELECT COUNT(*) FROM v_orders) % 2
    OFFSET (SELECT (COUNT(*) - 1) / 2 FROM v_orders));
-- Compare with the mean: the two differ because the mean is pulled down by the many small orders and up by the 120 order
SELECT ROUND(AVG(total), 2) AS mean_total FROM v_orders;

median_window
-------------
48.0         
(1 row)

median_offset
-------------
48.0         
(1 row)

mean_total
----------
42.73     
(1 row)



In [18]:
%%sql
WITH r AS (SELECT total, ROW_NUMBER() OVER (ORDER BY total, order_id) AS rn, COUNT(*) OVER () AS n FROM v_orders)
SELECT 'p25' AS pct, total FROM r WHERE rn = (n * 25 + 99) / 100
UNION ALL SELECT 'p50', total FROM r WHERE rn = (n * 50 + 99) / 100
UNION ALL SELECT 'p90', total FROM r WHERE rn = (n * 90 + 99) / 100
ORDER BY pct;

pct | total
----+------
p25 | 16.0 
p50 | 48.0 
p90 | 70.0 
(3 rows)



## 17. Cumulative distribution
- `CUME_DIST()` = share of rows with a value less than or equal to this one. `PERCENT_RANK()` = (rank - 1) / (rows - 1).
- Use it to answer "what share of orders are below 30?" or to draw a distribution curve.
- Tied values get the same `CUME_DIST` (it counts all peers).

In [19]:
%%sql
SELECT total, COUNT(*) AS orders,
       ROUND(100.0 * SUM(COUNT(*)) OVER (ORDER BY total) / SUM(COUNT(*)) OVER (), 1) AS cum_pct_by_hand
FROM v_orders GROUP BY total ORDER BY total;
SELECT order_id, total,
       ROUND(CUME_DIST() OVER (ORDER BY total), 3) AS cume_dist,
       ROUND(PERCENT_RANK() OVER (ORDER BY total), 3) AS pct_rank
FROM v_orders ORDER BY total, order_id;

total | orders | cum_pct_by_hand
------+--------+----------------
6.0   | 1      | 6.7            
13.0  | 1      | 13.3           
14.0  | 1      | 20.0           
16.0  | 1      | 26.7           
18.0  | 2      | 40.0           
35.0  | 1      | 46.7           
48.0  | 1      | 53.3           
50.0  | 1      | 60.0           
52.0  | 1      | 66.7           
56.0  | 1      | 73.3           
60.0  | 1      | 80.0           
65.0  | 1      | 86.7           
70.0  | 1      | 93.3           
120.0 | 1      | 100.0          
(14 rows)

order_id | total | cume_dist | pct_rank
---------+-------+-----------+---------
11       | 6.0   | 0.067     | 0.0     
10       | 13.0  | 0.133     | 0.071   
8        | 14.0  | 0.2       | 0.143   
5        | 16.0  | 0.267     | 0.214   
2        | 18.0  | 0.4       | 0.286   
14       | 18.0  | 0.4       | 0.286   
6        | 35.0  | 0.467     | 0.429   
13       | 48.0  | 0.533     | 0.5     
3        | 50.0  | 0.6       | 0.571   
15       | 52.0  | 0.

## 18. Moving top-seller: best product by trailing 3 months
- Build a calendar (recursive CTE), cross join it with products so every product has a row for every month (even with 0 sales), then use a window with `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` for the trailing 3-month units.
- Rank products inside each month and keep rank 1. Add the product id as a tiebreaker.
- Without the calendar the trailing window would skip empty months and cover the wrong period.

In [20]:
%%sql
WITH RECURSIVE cal(ym, n) AS (
    SELECT '2024-01', 1 UNION ALL
    SELECT strftime('%Y-%m', date(ym || '-01', '+1 month')), n + 1 FROM cal WHERE n < 12
),
monthly AS (
    SELECT cal.ym, p.id AS product_id, p.name,
           COALESCE((SELECT SUM(l.qty) FROM v_lines l WHERE l.product_id = p.id AND strftime('%Y-%m', l.order_date) = cal.ym), 0) AS units
    FROM cal CROSS JOIN products p
),
rolling AS (
    SELECT ym, product_id, name,
           SUM(units) OVER (PARTITION BY product_id ORDER BY ym ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS units_3m
    FROM monthly
),
ranked AS (
    SELECT ym, name, units_3m, ROW_NUMBER() OVER (PARTITION BY ym ORDER BY units_3m DESC, product_id) AS rn FROM rolling
)
SELECT ym, name AS top_seller, units_3m FROM ranked WHERE rn = 1 AND ym IN ('2024-03','2024-06','2024-09','2024-12') ORDER BY ym;

ym      | top_seller | units_3m
--------+------------+---------
2024-03 | Pen        | 2       
2024-06 | Cable      | 3       
2024-09 | Pen        | 10      
2024-12 | Pen        | 4       
(4 rows)



## 19. Budget vs actual (year-end variance)
- Variance = actual - budget. Variance % = variance / budget. For revenue, positive is favourable. For costs the sign flips, so store a flag or a sign per line.
- Use a `LEFT JOIN` from the budget table so budget lines with no actuals still show (and `COALESCE` the actual to 0).
- End with a total row. A total variance % is total variance / total budget, not the average of the line percentages.

In [21]:
%%sql
WITH act AS (
    SELECT category, SUM(revenue) AS actual FROM v_lines WHERE year = 2024 GROUP BY category
),
v AS (
    SELECT b.category, b.budget, COALESCE(a.actual, 0) AS actual, COALESCE(a.actual, 0) - b.budget AS variance
    FROM budget_2024 b LEFT JOIN act a ON a.category = b.category
)
SELECT * FROM (
SELECT 0 AS o, category, budget, actual, variance, ROUND(100.0 * variance / budget, 1) AS variance_pct,
       CASE WHEN variance >= 0 THEN 'favourable' ELSE 'adverse' END AS status
FROM v
UNION ALL
SELECT 1, 'TOTAL', SUM(budget), SUM(actual), SUM(variance), ROUND(100.0 * SUM(variance) / SUM(budget), 1),
       CASE WHEN SUM(variance) >= 0 THEN 'favourable' ELSE 'adverse' END
FROM v
) ORDER BY o, category;

o | category   | budget | actual | variance | variance_pct | status    
--+------------+--------+--------+----------+--------------+-----------
0 | Home       | 150.0  | 76.0   | -74.0    | -49.3        | adverse   
0 | Stationery | 60.0   | 62.0   | 2.0      | 3.3          | favourable
0 | Tech       | 400.0  | 290.0  | -110.0   | -27.5        | adverse   
1 | TOTAL      | 610.0  | 428.0  | -182.0   | -29.8        | adverse   
(4 rows)



## 20. A mini executive summary: one query, several CTEs
- Each CTE answers one question. The final query stitches the answers into one row per year, ready to paste into a report.
- Pattern: `kpi` (totals), `top_cat` (best category per year with `ROW_NUMBER`), `top_cust` (best customer per year), then join on `year`.
- Keep CTEs small and named after what they return. It reads like a report outline.

In [22]:
%%sql
WITH kpi AS (
    SELECT year, COUNT(*) AS orders, COUNT(DISTINCT customer_id) AS customers, SUM(total) AS revenue,
           ROUND(1.0 * SUM(total) / COUNT(*), 1) AS aov
    FROM v_orders GROUP BY year
),
cat AS (
    SELECT year, category, SUM(revenue) AS rev,
           ROW_NUMBER() OVER (PARTITION BY year ORDER BY SUM(revenue) DESC, category) AS rn
    FROM v_lines GROUP BY year, category
),
cust AS (
    SELECT o.year, c.name, SUM(o.total) AS spend,
           ROW_NUMBER() OVER (PARTITION BY o.year ORDER BY SUM(o.total) DESC, c.name) AS rn
    FROM v_orders o JOIN customers c ON c.id = o.customer_id GROUP BY o.year, c.id
)
SELECT k.year, k.orders, k.customers, k.revenue, k.aov,
       ROUND(100.0 * (k.revenue - LAG(k.revenue) OVER (ORDER BY k.year)) / LAG(k.revenue) OVER (ORDER BY k.year), 1) AS growth_pct,
       t.category AS top_category, ROUND(100.0 * t.rev / k.revenue, 1) AS top_category_pct,
       u.name AS top_customer, u.spend AS top_customer_spend
FROM kpi k
JOIN cat  t ON t.year = k.year AND t.rn = 1
JOIN cust u ON u.year = k.year AND u.rn = 1
ORDER BY k.year;

year | orders | customers | revenue | aov  | growth_pct | top_category | top_category_pct | top_customer | top_customer_spend
-----+--------+-----------+---------+------+------------+--------------+------------------+--------------+-------------------
2023 | 6      | 5         | 213.0   | 35.5 | NULL       | Home         | 62.0             | Asha         | 83.0              
2024 | 9      | 5         | 428.0   | 47.6 | 100.9      | Tech         | 67.8             | Fay          | 168.0             
(2 rows)

